# [ベースライン advanced 版] 第1回データ分析コンペティション：NFL Draft Prediction

　本ノートブックでは、basic 版で扱った内容をベースに、機械学習コンペティションでよく行われる「改善サイクル」を体験します。

　単に 1 つモデルを学習して提出するのではなく、

- **EDA（探索的データ分析）** をもとに前処理方針を考える
- ベースラインを作って過学習を意識する
- クロスバリデーションで評価の信頼度を上げる
- 仮説に基づいて特徴量を追加して比較する
- 複数のモデルを比較する
- 予測をアンサンブルする

という流れをたどることで、 **「動かす」から「考えて改善する」へ進む** ための橋渡しを目指します。

## 目次
0. ライブラリ・データ読み込み
1. データの概観・EDA
2. 前処理
3. ベースラインモデルの構築
4. 5-fold クロスバリデーション
5. 特徴量エンジニアリング
6. 様々なモデルの構築・比較
7. モデルのアンサンブリング
8. LightGBM のハイパーパラメータチューニング
9. 予測の出力・提出
10. 今後のステップ

## 0. ライブラリ・データ読み込み

In [ ]:
# Google Driveのマウント
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# モジュールのインポート
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import roc_auc_score

import lightgbm as lgb

import warnings
warnings.filterwarnings('ignore')

### データ配置

このノートブックではGoogle Drive内において、以下のデータ配置を想定しています。

```
マイドライブ/
└── GCI/
    └── competition_1/
        ├── baseline_advanced.ipynb  # このファイル
        └── data/
            ├── train.csv
            ├── test.csv
            └── sample_submission.csv
```

In [ ]:
# データの読み込み
PATH = '/content/drive/My Drive/GCI/competition_1/data/'

train = pd.read_csv(PATH + 'train.csv')
test = pd.read_csv(PATH + 'test.csv')

print('Train:', train.shape)
print('Test:', test.shape)

## 1. データの概観・EDA

　まずは EDA を行います。EDA とは "Exploratory Data Analysis" の略で、モデルを作る前にデータをよく観察し、その特徴を理解する作業のことです。

　EDA は単にデータを眺めるだけではなく、**後の作業（前処理・特徴量設計・モデル選び）で、どう判断するかの材料を集める** ために行います。具体的には、次のような観点でデータを確認していきます。

- **欠損やノイズはどこにあるか**：欠損が多い列が分かれば、後の前処理で「平均で補完する」「列ごと削除する」などの方針を決められます。
- **数値の分布はどうなっているか**：列ごとに値の桁が大きく違うようなら、後で値の大きさを揃える前処理（スケーリング）が必要かどうかを判断できます。また、極端に外れた値があれば、それへの対処も検討します。
- **カテゴリと目的変数にどんな関係があるか**：たとえば「どのカテゴリの選手が指名されやすいか」が見えると、効きそうな特徴量に当たりがつき、文字列を数値にどう変換するか（エンコーディング）の方針も立てやすくなります。
- **目的変数の分布は偏っていないか**：偏りが強いと、モデルが「多数派クラスばかり予測しておけば当たる」という安易な学習に陥ることがあります。また、評価指標との相性も意識する必要があります。

　次の各ステップで、これらを順番に確認していきます。

In [ ]:
train.head()

In [ ]:
train.info()

　欠損値を確認します。多くの機械学習モデルは欠損値をそのまま入力できないため、**どの列にどれくらい欠損があるか** を把握しておくことは、後の前処理（補完するか、欠損自体を新しい特徴量にするか、列ごと削除するか）の判断材料になります。

In [ ]:
train.isnull().sum()

In [ ]:
test.isnull().sum()

　目的変数 `Drafted` の分布を確認します。「Drafted=1 と Drafted=0 のどちらがどれだけ多いか」を **クラスバランス** と呼びます。

　クラスバランスを把握しておきたい理由は、データが偏っていると **モデルが「とりあえず多い方を予測しておけば当たる」という安易な学習** に陥ることがあるためです。たとえば 99 対 1 のデータなら、全部「多い方」と予測するだけで accuracy 99% に見えてしまいます。

In [ ]:
f, ax = plt.subplots(1, 2, figsize=(16, 6))
train['Drafted'].value_counts().plot.pie(autopct='%1.1f%%', ax=ax[0], shadow=True)
ax[0].set_title('Drafted')
ax[0].set_ylabel('')
sns.countplot(x='Drafted', data=train, ax=ax[1])
ax[1].set_title('Drafted')
plt.show()

　今回は約 65% : 35% で、強い不均衡ではないものの、わずかに偏っています。**今回のコンペの評価指標は AUC** です。AUC は Area Under the ROC Curve の略で、ROC 曲線の下側の面積を表します。スコアは 0.5 がランダム予測と同じ、1.0 が完璧な分類器を意味します。AUC は不均衡データでもスコアが歪みにくい性質を持つので、このデータでも問題なく使えます。ちなみに、もし accuracy で評価していたら「全部 1 と予測する」だけで 65% を達成できてしまうため、AUC が選ばれているのは妥当な判断と言えます。

　数値特徴量の分布をヒストグラムで確認します。見るポイントは次の 3 つです。

1. **分布の形**：左右対称のベル型に近い「正規分布」のような形か、それとも片側に偏った歪んだ形か
2. **外れ値の有無**：他の値から極端に離れた値はないか
3. **列ごとのスケールの違い**：列によって値の桁が大きく違わないか（例: `Year` は 2010 前後、`Sprint_40yd` は 5 前後、`Height` は 1.85 前後）

　これらを **どこまで気にするかは使うモデルによって変わります**。

- **線形モデル・ニューラルネット**：分布の形やスケールを揃える前処理が効果的です。代表的なのが「標準化」と呼ばれる手法。スケールが揃っていないと「値が大きい列に過剰に引っ張られる」現象が起きるためです。
- **決定木系（RandomForest や LightGBM）**：「身長 > 1.85m？」のように **値の大小ではなく閾値で分岐** するため、スケールの違いはほぼ気にしなくて済みます。

　「とりあえず全部スケーリングする」のではなく、**使うモデルに応じて必要な前処理を判断する** のが、効率よくスコアを伸ばすための実践的な感覚です。

In [ ]:
numeric_cols = train.select_dtypes(include=['number']).columns.drop(['Id', 'Drafted'])

cols = 3
rows = (len(numeric_cols) + cols - 1) // cols
plt.figure(figsize=(5 * cols, 4 * rows))

for i, col in enumerate(numeric_cols, 1):
    plt.subplot(rows, cols, i)
    plt.hist(train[col].dropna(), bins=30, edgecolor='black')
    plt.title(f'Histogram of {col}')
    plt.xlabel(col)
    plt.ylabel('Frequency')

plt.tight_layout()
plt.show()

　数値特徴量どうしの相関をヒートマップで確認します。相関を見る目的は主に 2 つあります。

1. **冗長な情報の発見**：相関が極端に高い列が複数あるとき、それらは似た情報を重複して持っていると考えられます。たとえば「身長(m)」と「身長(cm)」のような列が両方あると、片方で十分です。線形モデルではこの重複が **多重共線性** という問題を引き起こします。これは似た列が複数あることで各列の係数が不安定になる現象で、解釈や予測の安定性に影響します。一方、決定木系ではこの問題は起きにくいです。
2. **特徴量エンジニアリングのヒント**：強い負の相関や予想と違う関係が見つかったら、ドメイン知識を確認するきっかけになります（後ほど Sprint_40yd でこの例を見ます）。

In [ ]:
# 数値列の相関行列
df_numeric = train.select_dtypes(include=['number']).drop(columns=['Id', 'Drafted'])

plt.figure(figsize=(12, 10))
sns.heatmap(df_numeric.corr(), annot=True, fmt='.2f', cmap='coolwarm',
            vmin=-1, vmax=1, square=True, linewidths=0.5)
plt.title('Correlation Matrix of Numeric Features', fontsize=16)
plt.show()

　`Agility_3cone` と `Shuttle` には強い正の相関があることが分かります。これらは似た筋肉や動作を使うテストなので、納得しやすい結果です。

　一方、`Sprint_40yd` と `Broad_Jump` には強い負の相関が見られます。どちらも脚力を使う種目なのに、なぜ逆向きの関係になるのでしょうか？ `Sprint_40yd` の値の分布を箱ひげ図で確認してみましょう。

In [ ]:
plt.figure(figsize=(6, 3))
sns.boxplot(x=train['Sprint_40yd'])
plt.title('Box Plot of Sprint_40yd', fontsize=16)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

　`Sprint_40yd` の値はおおよそ 4.25 〜 6.00 の範囲に分布しています。これは 40 ヤード（約 36 m）走の **タイム（秒）** を表していると考えられます。

　つまり、`Sprint_40yd` は **値が小さいほど速い＝パフォーマンスが高い** という、他の数値特徴量とは逆向きの解釈をする列です。これがヒートマップで負の相関として現れていたわけです。

　このように、その分野の専門知識を **ドメイン知識** と呼びます。今回なら NFL や陸上競技の知識のことです。ドメイン知識と可視化を結びつけて解釈することが、EDA で得られる大事な気づきになります。

　次にカテゴリ特徴量を確認します。各カテゴリ列がどんな値を持っているか、また目的変数とどんな関係があるかを順に見ていきます。

In [ ]:
categorical_cols = train.select_dtypes(include=['object', 'category']).columns
for col in categorical_cols:
    print(f"{col}: {train[col].nunique()} levels")

　`School` は水準数が非常に多いので、ここでは他のカテゴリ変数を可視化します。

In [ ]:
plot_cols = [c for c in categorical_cols if c != 'School']

fig, axes = plt.subplots(1, len(plot_cols), figsize=(5 * len(plot_cols), 5))
if len(plot_cols) == 1:
    axes = [axes]

for i, col in enumerate(plot_cols):
    sns.countplot(x=col, data=train, order=train[col].value_counts().index, ax=axes[i])
    axes[i].set_title(f'Count Plot of {col}')
    axes[i].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

　各カテゴリの `Drafted` 平均（指名されている割合）を確認します。これは **カテゴリ変数が目的変数に効きそうかを判断する最も基本的な可視化** で、水準ごとの平均が大きく異なれば、そのカテゴリ変数は予測に役立つ情報を持っていることが分かります。逆に水準ごとの差が小さければ、その変数の予測力は限定的だと判断できます。

In [ ]:
fig, axes = plt.subplots(1, len(plot_cols), figsize=(5 * len(plot_cols), 5))
if len(plot_cols) == 1:
    axes = [axes]

for i, col in enumerate(plot_cols):
    mean_values = train.groupby(col)['Drafted'].mean().sort_values(ascending=False)
    sns.barplot(x=mean_values.index, y=mean_values.values, ax=axes[i])
    axes[i].set_title(f'Average Drafted by {col}')
    axes[i].set_ylim(0, 1)
    axes[i].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## 2. 前処理

　EDA の結果を踏まえて、ここではモデルに学習させられる形にデータを整えます。

　**なぜ前処理が必要か？** 機械学習モデルの多くは、文字列のままや欠損があるデータを直接受け取れません。また、モデルにとって役に立たない列をそのまま渡すとノイズになり、かえって性能を下げる原因にもなります。**「データを使える形に整える」「ノイズになる情報を除く」** ことで、モデルが本来の力を発揮できるようにするのが前処理の目的です。

　今回は次の方針で進めます。それぞれを選んだ理由も併記します。

- **`Id` を削除**：単なる識別番号で予測に意味のある情報を含まないためです。逆にこれを残すと、ツリー系モデルが Id を「分岐の手がかり」として使ってしまい、過学習につながる可能性があります。
- **`School` を削除**：水準数が 200 超と非常に多く、各水準あたりのサンプル数が少ないため、そのまま入れると逆にノイズになります。本格的にスコアを上げるなら **Target Encoding** などで情報を活かせます。これは水準ごとに目的変数の平均値を計算して、その値で置き換える手法です。
- **数値の欠損は train の平均で補完**：欠損値を入れたままだと多くのモデルが学習できないため、何らかの値で埋める必要があります。今回はシンプルに平均値を使いますが、中央値・グループ別平均・欠損自体を新しい特徴量にする等、選択肢は多様です。なお、test 自体の値も使って train+test 全体の平均で補完するという選択肢もあり、コンペルール上は許容されています。ただし実務感覚に近づけるため、本ノートブックでは train の情報のみで処理します。
- **カテゴリ変数は Label Encoding**：文字列のままでは多くのモデルが扱えないため数値化が必要です。ここでは整数 ID に置き換える Label Encoding を使います。

  たとえば `Position_Type` 列に `kicking_specialist`、`offensive_lineman`、`backs_receivers` の 3 種類があるとき、Label Encoding はそれぞれを `0`、`1`、`2` のような整数に置き換えます。

  このとき **決定木系のモデル（RandomForest、LightGBM）** は「値が 1 以上なら右の枝、1 未満なら左の枝」のように **閾値で分岐** するため、整数の大小がそのまま出てきても問題はありません。割り当てられた数値の順序は、分岐の境界として使われるだけです。

  一方、**線形モデル（LogisticRegression）** は内部で `予測値 = w₁ × Position_Type_ID + w₂ × Age + ...` のように **特徴量に重みを掛けて足し算** します。この計算上、`backs_receivers (=2)` は `offensive_lineman (=1)` の 2 倍、`kicking_specialist (=0)` の無限倍として扱われ、本来は順序のないカテゴリに勝手に大小関係をつけて学習してしまいます。実質的に `kicking_specialist < offensive_lineman < backs_receivers` という意味のない順序が入ってしまうわけです。

  これを避けるには、カテゴリごとに 0/1 のフラグ列を作る **One-Hot Encoding** の方が適しています。たとえば `Position_Type` 列を `is_kicking_specialist`、`is_offensive_lineman`、`is_backs_receivers` の 3 列に分け、該当するカテゴリだけ 1、他は 0 にする方法です。こうすれば各カテゴリが独立した 0/1 として扱われ、勝手な順序付けが起きません。

  今回は決定木系を主に使う前提で Label Encoding を採用します。

  なお、`fit` 時には `pd.concat([train[c], test[c]])` で train と test を結合してから対応関係を学習しています。これは **test にだけ存在するカテゴリ** があった場合、train だけで学習した変換器ではそのカテゴリを変換できずエラーになるためです。両方を結合してから学習することでこの問題を防ぎます。

In [ ]:
train = train.drop(columns=["Id", "School"])
test = test.drop(columns=["Id", "School"])

cols_to_fill = [
    'Age', 'Sprint_40yd', 'Vertical_Jump', 'Bench_Press_Reps',
    'Broad_Jump', 'Agility_3cone', 'Shuttle'
]
for col in cols_to_fill:
    mean_value = train[col].mean()
    train[col] = train[col].fillna(mean_value)
    test[col] = test[col].fillna(mean_value)

label_encoders = {}
for c in ["Player_Type", "Position_Type", "Position"]:
    label_encoders[c] = LabelEncoder()
    label_encoders[c].fit(pd.concat([train[c], test[c]]).astype(str))
    train[c] = label_encoders[c].transform(train[c].astype(str))
    test[c] = label_encoders[c].transform(test[c].astype(str))

train.head()

## 3. ベースラインモデルの構築

　まずは **ベースラインモデル** を作ります。ベースラインモデルとは、これから工夫を重ねていく上での **基準点となるシンプルなモデル** のことです。「特徴量を追加したら何点上がった」「モデルを変えたら何点上がった」を測るには、最初に **基準となるスコア** が必要です。

　ここでは basic 版と同様にランダムフォレストを使います。ただし advanced 版では、**過学習** を意識して、学習データと検証データの両方で AUC を見ます。

　また、ここでは `max_depth=10`、`min_samples_leaf=1` という、**あえて複雑な設定** で baseline を作ります。これは木の深さの上限が十分大きく、葉に 1 サンプルだけになっても分岐を続けるという制約の緩い設定で、**学習データに細かく合わせられる分、過学習が起きやすい状態** です。basic 版で使った `max_depth=5` と比べて、Train AUC と Valid AUC の差がどれだけ広がるか、過学習が起きる様子を意図的に観察するところから始めます。

In [ ]:
X = train.drop(columns=["Drafted"])
y = train["Drafted"]

X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

rfc = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    min_samples_leaf=1,
    random_state=42,  # 乱数固定（再現性のため）
    n_jobs=-1         # 全 CPU コアで並列計算（学習を速くする）
)
rfc.fit(X_train, y_train)

train_pred = rfc.predict_proba(X_train)[:, 1]
valid_pred = rfc.predict_proba(X_valid)[:, 1]

print('Train AUC:', round(roc_auc_score(y_train, train_pred), 4))
print('Valid AUC:', round(roc_auc_score(y_valid, valid_pred), 4))

　Train AUC と Valid AUC を見比べると、Train 側がかなり高く、Valid 側との差が大きいことが分かります。これは **過学習（overfitting）** が起きているサインです。

　**過学習をテスト勉強にたとえる** と、過去問を丸暗記してしまった生徒のような状態です。過去問、つまり学習データには完璧に答えられるけれど、本番で出る初見の問題、つまり未知のデータにはうまく対応できません。モデルが学習データの「本質的な傾向」だけでなく、たまたまそのデータに含まれていた **偶然のパターン、いわゆるノイズ** まで覚え込んでしまうと、こうしたことが起きます。

　なぜ過学習するのか？　それは **モデルの「複雑さ」、つまり表現力が高すぎる** ときです。複雑なモデルは学習データの細部まで合わせる余地があるので、ノイズも一緒に覚えてしまいます。逆に単純すぎるモデルは、そもそも学習データの傾向を捉えきれません。

　この「単純すぎる」と「複雑すぎる」のあいだのトレードオフは **バイアス・分散トレードオフ** と呼ばれます。先ほどのテスト勉強の比喩を続けると分かりやすいです。

- **モデルが単純すぎる**：たとえば「どんな問題でも全部 "Yes" と答える」と決めて挑む生徒のようなものです。問題ごとの違いを見ずに同じ答えを出すので、**毎回だいたい同じくらい的外れ** な結果になります。学習データが変わってもこの傾向は変わらず、真の答えから系統的にズレ続けます。これを **バイアスが大きい** と言います。
- **モデルが複雑すぎる**：過去問の文言を一字一句まで丸暗記した生徒のようなものです。学習した過去問とそっくりな問題なら満点ですが、本番では **少し文言が変わるだけで全く答えられません**。さらに、勉強した過去問の組み合わせが少し違うだけで本番の点数が大きくブレてしまい、結果が不安定になります。これを **分散が大きい** と言います。

　ちょうどよい複雑さのモデルは、過去問から **本質的な解き方のパターン** を学び取り、初見の問題にも応用できる生徒のような状態です。**ズレも小さく、結果も安定して高得点が出せる** モデルを見つけるのが、モデリングの中心的な仕事になります。

　ランダムフォレストでは、木の深さの上限を表す `max_depth` や、葉に入る最小サンプル数を表す `min_samples_leaf` が「複雑さ」を制御するハイパーパラメータです。これらを動かして過学習を抑える方向に調整してみましょう。

In [ ]:
param_grid = {
    'max_depth': [3, 5, 7],
    'min_samples_leaf': [1, 5, 20]
}

for max_depth in param_grid['max_depth']:
    for min_samples_leaf in param_grid['min_samples_leaf']:
        rfc_grid = RandomForestClassifier(
            max_depth=max_depth,
            min_samples_leaf=min_samples_leaf,
            n_estimators=100,
            n_jobs=-1,
            random_state=42
        )
        rfc_grid.fit(X_train, y_train)
        train_pred = rfc_grid.predict_proba(X_train)[:, 1]
        valid_pred = rfc_grid.predict_proba(X_valid)[:, 1]
        print(f'max_depth: {max_depth}, min_samples_leaf: {min_samples_leaf}')
        print('    Train AUC: {}, Valid AUC: {}'.format(
            round(roc_auc_score(y_train, train_pred), 4),
            round(roc_auc_score(y_valid, valid_pred), 4)
        ))

　結果を見ると、**`max_depth` を小さく（木を浅く）するほど** Train/Valid のギャップが縮まり、過学習が緩和されていることが分かります。同じように、**`min_samples_leaf` を大きく（葉に最低必要なサンプル数を増やす）するほど** 、細かい分岐が抑えられて過学習が緩和される傾向が見えるはずです。どちらも「モデルの複雑さを抑える」方向の調整で、過学習を抑える効き方は似ています。

　ただし、ギャップが小さい設定が必ずしも Valid AUC が高いわけではありません。**過学習の抑制** と **Valid AUC** の両方を見比べて、バランスのよい設定を選ぶ必要があります。本ノートブックでは以降の章で `max_depth=5, min_samples_leaf=5` を使います。Train/Valid のギャップが比較的小さく、過学習を抑えつつスコアも維持できる設定です。

　もちろん別の組み合わせを試してみるのも良い練習になります。

## 4. 5-fold クロスバリデーション

　`train_test_split` は手軽ですが、1 回しか分割していないので **たまたまその分割で高い／低い** ということが起こりえます。1 回しか試験を受けていないと、本当の実力か運だったか分かりません。

　より信頼度の高い評価方法が **クロスバリデーション（CV）** です。データを K 個のグループ（fold）に分けて、**「1 つを検証用、残りを学習用」を K 回繰り返します**。同じデータを K 通りの分け方で評価して平均を取るイメージです。今回は K=5 の **5-fold CV** を使います。

　ここで使う `StratifiedKFold(n_splits=5, shuffle=True, random_state=42)` の各設定の意味と意図は次のとおりです。

- **`n_splits=5`**：fold の数。多いほど評価が安定するが計算量が増える。3〜10 が一般的で、5 はバランスのよい標準値
- **`Stratified`（層化）**：fold ごとに目的変数の比率を揃える。クラス不均衡データではこれをしないと fold 間で評価がブレやすい
- **`shuffle=True`**：データの並び順を無作為化してから分割します。ただし、**時系列データには使えない** 点に注意してください。時系列では過去のデータで学習して未来を予測するのが前提なので、無作為に並び替えると話が崩れてしまい、別の分割方法が必要になります
- **`random_state=42`**：乱数の種を固定。何回実行しても同じ分割になるので、結果の再現性が保たれる

In [ ]:
# 5-fold CV の分割設定（以降の章でも同じ分割を使う）
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

rfc_cv = RandomForestClassifier(
    n_estimators=100,
    max_depth=5,
    min_samples_leaf=5,
    random_state=42,
    n_jobs=-1
)

cv_scores = cross_val_score(rfc_cv, X, y, cv=skf, scoring='roc_auc')
print('CV AUC Scores:', np.round(cv_scores, 4))
print('Mean CV AUC:', round(cv_scores.mean(), 4))

　fold ごとのばらつきと、その平均 AUC が分かりました。この **5 fold で算出した平均 AUC** が、このデータでの基準スコアになります。

　以降このノートブックでは、この値を **CV AUC**（または **CV スコア**）と呼びます。「特徴量を追加したら CV AUC が上がった／下がった」「モデルを変えたら CV AUC が変わった」という形で、改善の効果を測る基準として使っていきます。

## 5. 特徴量エンジニアリング

　Section 4 までで、このデータでの基準スコア（CV AUC）が分かりました。**ここからは、その基準スコアをさらに上げるための工夫** を試していきます。改善の打ち手はいくつかありますが、まず取り組むのが **特徴量エンジニアリング** です。

　特徴量エンジニアリングとは、元のデータから **モデルがより学習しやすい新しい列を作って加える** 作業のことです。仮説を立てて新しい特徴量を作り、CV スコアが上がるかどうかを検証します。スコアを伸ばすうえで最も基本的かつ効果が出やすい手段の 1 つです。

　**そもそも、なぜ特徴量を作るだけでスコアが上がるのか？**　「LightGBM のような強いモデルなら、列の組み合わせも自動で見つけてくれそう」と思うかもしれません。確かに決定木系は特徴量どうしの掛け合わせをある程度学習できますが、限界もあります。

- **データ量が少ないと自動発見しづらい**：「身長が高くて体重も重い選手は指名されやすい」のような関係も、サンプルが少ないと統計的に拾えません。
- **割り算・足し算は決定木が苦手**：決定木は「身長 > 1.85m？」のように **1 列ごとに分岐** するため、`BMI = 体重 / 身長²` のような **複数列を組み合わせた値** を直接は表現できません。事前に列として渡しておけば、その値で分岐できるようになります。
- **ドメイン知識を渡せる**：人間が「この組み合わせには意味がある」と知っている情報を、列として明示的に渡すことでモデルの学習が効率化します。

　つまり特徴量エンジニアリングは、**モデルが自力では見つけにくい関係を、人間が先に列として渡してあげることで、スコアを伸ばす** 作業なのです。

　今回は NFL Draft のドメインを意識しながら、スコア向上を狙って 3 つの特徴量を試します。

- **`BMI = 体重 / 身長²`**：体格バランスの指標。同じ身長でも筋肉質と痩せ型で違う、という情報を持つ
- **`SpeedScore = 体重 / Sprint_40yd`**：体重がある選手で 40 ヤード走が速いほど高い値になります。なお Sprint_40yd は走行タイムなので小さいほど速いことに注意してください。「重いのに速い＝身体能力が高い」という直感を数値化したもの
- **`Explosion = Vertical_Jump + Broad_Jump`**：垂直跳びと立ち幅跳びの合計。瞬発力を 2 方向で総合的に表す

　いずれも仮説段階で、効くかどうかは CV スコアで検証してみないと分かりません。

In [ ]:
# 元の train, test を変更しないように copy() で別オブジェクトを作る
df_fe = train.copy()
df_fe_test = test.copy()

for target_df in [df_fe, df_fe_test]:
    target_df['BMI'] = target_df['Weight'] / (target_df['Height'] ** 2)
    target_df['SpeedScore'] = target_df['Weight'] / target_df['Sprint_40yd']
    target_df['Explosion'] = target_df['Vertical_Jump'] + target_df['Broad_Jump']

X_fe = df_fe.drop(columns=["Drafted"])
y_fe = df_fe["Drafted"]
X_fe_test = df_fe_test.copy()

rfc_fe = RandomForestClassifier(
    n_estimators=100,
    max_depth=5,
    min_samples_leaf=5,
    random_state=42,
    n_jobs=-1
)

cv_scores_fe = cross_val_score(rfc_fe, X_fe, y_fe, cv=skf, scoring='roc_auc')
print('Feature Engineering CV AUC Scores:', np.round(cv_scores_fe, 4))
print('Mean CV AUC:', round(cv_scores_fe.mean(), 4))

　平均 AUC が変化したことが確認できると思います。

　ここでのポイントは、**特徴量を追加すれば必ずスコアが上がるわけではない** ということです。良くなる仮説もあれば、効かない／むしろ下がる仮説もあります。

　大切なのは、**「仮説を立てる、特徴量を追加する、スコアを検証する、効けば残して効かなければ戻す」というサイクル** を回すことです。実際のコンペでは、こうした地道な検証の積み重ねがスコア向上につながります。

## 6. 様々なモデルの構築・比較

　特徴量エンジニアリングの次は、**モデル自体を変えてスコアを上げられないか** を試します。これまでは RandomForest 1 種類だけを使ってきましたが、データの性質によって得意なモデルは違うので、**複数のモデルを比較して、このデータに最も合うモデルを見つける** のが狙いです。

　今回は次の 3 つのモデルを使います。これらは **学習の仕組みが根本から違う 3 つの代表的なモデル** で、性質の違うものから 1 つずつ選んだ形になっています。

- **LightGBM（勾配ブースティング）**：弱いモデル（浅い決定木）を順番に重ねて、前のモデルの間違いを次のモデルが修正していく仕組み。表形式データのコンペでは事実上の標準で、その理由は ① スコアが出やすい、② 学習が速い、③ 欠損値・カテゴリを扱いやすい、④ 比較的小さいデータでも安定する、といった性質が揃っているためです。
- **LogisticRegression（線形モデル）**：特徴量の線形結合に **sigmoid 関数** をかけて確率を出す古典的な手法です。sigmoid 関数は任意の実数を 0〜1 の値に押し込む S 字型の関数で、これによって出力を確率として扱えるようになります。表現力は限定的ですが **解釈性が高く**、各特徴量がどう効いたか係数で読み取れます。データが線形に近い構造を持つときには非常に強力です。
- **MLPClassifier（ニューラルネット）**：**多層パーセプトロン** とも呼ばれる、ニューラルネットの基本形です。入力 → 中間層 → 出力 と層を重ねて、各層で重みつき和と非線形変換を繰り返す構造になっています。非線形な関係や特徴量の複雑な相互作用を捉えられる柔軟性が魅力ですが、**チューニングが繊細** で、データが小さいと過学習しやすい弱点があります。

　仕組みの違うモデルをわざわざ並べて試すのには 2 つの意味があります。1 つは **「No Free Lunch（万能な手法は存在しない）」** という機械学習の原則。データの性質によって得意なモデルは違うので、複数試して初めて「このデータにはこれが向いている」が分かります。もう 1 つは、**互いに違う傾向の予測** を作っておくことで、後の **アンサンブル** で組み合わせる選択肢が広がる点です。

　ここで一点注意があります。LogisticRegression と MLPClassifier は **特徴量のスケール（値の大小）に敏感** です。たとえば `Year` は 2010 前後、`Sprint_40yd` は 5 前後、`Height` は 1.85 前後というように、列ごとに値の規模が大きく異なります。このままだと「値が大きい列ほど影響が強い」と誤って学習してしまい、性能が大きく下がります。

　そこで、これらのモデルには **標準化** を前段に挟みます。標準化とは各列を平均 0・標準偏差 1 に揃える処理で、sklearn の `StandardScaler` で行えます。

　ここで使う **`Pipeline`** は、前処理とモデルを 1 つの処理の流れとしてまとめる sklearn の仕組みです。Pipeline を使うと、CV の各 fold ごとに「学習データだけで標準化のパラメータ、つまり平均と標準偏差を計算し、検証データにはそれを適用するだけ」という処理が自動的に行われます。これによって、**学習データと検証データの間で情報が漏れないように前処理を組み込む** ことができます。手動で書くと検証データの平均値まで使ってしまい、データリークが起きやすい部分です。

　一方、LightGBM や RandomForest のような決定木系のモデルは **値の大小ではなく順序で分岐する** ため、スケーリングは不要です。

In [ ]:
lgb_model = lgb.LGBMClassifier(
    n_estimators=200,
    learning_rate=0.05,
    num_leaves=31,
    random_state=42,
    n_jobs=-1,
    verbose=-1
)

# LR と MLP は標準化を前段に挟む（Pipeline で CV ごとに正しく fit/transform される）
lr_model = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(max_iter=1000, random_state=42))
])

mlp_model = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', MLPClassifier(hidden_layer_sizes=(100, 50), random_state=42, max_iter=500))
])

models = {
    'LightGBM': lgb_model,
    'LogisticRegression': lr_model,
    'MLPClassifier': mlp_model
}

for name, model in models.items():
    scores = cross_val_score(model, X_fe, y_fe, cv=skf, scoring='roc_auc')
    print(f'{name}: mean CV AUC = {scores.mean():.4f}')

　モデルによって AUC が大きく変わることが分かります。今回は LightGBM が他を上回るスコアを出しましたが、これは **データの性質と相性が良かった** ためで、いつも LightGBM が勝つわけではありません。

- **LightGBM が勝ちやすい場面**：表形式の中規模データ（数千〜数百万行）、特徴量どうしの非線形な相互作用が重要、欠損値や異なるスケールが混在
- **線形モデルが勝ちやすい場面**：データが少ない（過学習リスクが大）、特徴量が線形に並んでいる、解釈性が必要
- **ニューラルネットが勝ちやすい場面**：データが大量（数十万〜数百万件）、画像・テキスト・系列など構造化された入力、十分なチューニング工数がある

　今回の NFL Draft データは「表形式・約 2400 行・スポーツテストの数値が中心」なので、勾配ブースティングが強いのは納得できる結果です。**データの性質に合うモデルを選ぶ** という意識を持つことが、モデル選定の出発点になります。

## 7. モデルのアンサンブリング

　Section 6 で 3 つのモデルを比較しました。次は、**それらを組み合わせてさらにスコアを上げられないか** を試します。これを **アンサンブル** と呼びます。

　アンサンブルとは、複数モデルの予測を組み合わせて最終予測とする手法です。たとえば 3 人の予報士に天気を聞いて多数決を取ると、1 人だけに聞くより当たりやすくなりますよね。これと同じ発想で、モデルも複数の意見を組み合わせると、それぞれの間違いが平均化されて予測が安定しやすくなります。

　ここでは LightGBM・LogisticRegression・MLP の 3 モデルの予測確率を **単純平均** して最終予測にします。

　なお、ここでは Section 6 と異なり、`cross_val_score`（簡単な評価用）ではなく **自前で `StratifiedKFold` を回します**。これは fold ごとに **OOF（Out-Of-Fold）予測** ＝「その fold で学習に使われていないデータに対する予測」と、test 予測の両方を保存して、アンサンブル予測を作る必要があるためです。

　また、コード中の `test_lgb += lgb_model.predict_proba(X_fe_test)[:, 1] / skf.n_splits` という書き方にも注目してください。これは **fold ごとの test 予測を `1/n_splits（=1/5）` ずつ加算** することで、最終的に **5 fold の平均 test 予測** を作る、よく使われる書き方です。各 fold は少しずつ違う 4/5 のデータで学習されているので、それらの予測を平均することで単一モデルより安定した予測が得られます。

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

oof_lgb = np.zeros(len(X_fe))
oof_lr = np.zeros(len(X_fe))
oof_mlp = np.zeros(len(X_fe))

test_lgb = np.zeros(len(X_fe_test))
test_lr = np.zeros(len(X_fe_test))
test_mlp = np.zeros(len(X_fe_test))

for fold, (tr_idx, va_idx) in enumerate(skf.split(X_fe, y_fe)):
    X_tr, X_va = X_fe.iloc[tr_idx], X_fe.iloc[va_idx]
    y_tr, y_va = y_fe.iloc[tr_idx], y_fe.iloc[va_idx]

    lgb_model.fit(X_tr, y_tr)
    lr_model.fit(X_tr, y_tr)
    mlp_model.fit(X_tr, y_tr)

    oof_lgb[va_idx] = lgb_model.predict_proba(X_va)[:, 1]
    oof_lr[va_idx]  = lr_model.predict_proba(X_va)[:, 1]
    oof_mlp[va_idx] = mlp_model.predict_proba(X_va)[:, 1]

    test_lgb += lgb_model.predict_proba(X_fe_test)[:, 1] / skf.n_splits
    test_lr  += lr_model.predict_proba(X_fe_test)[:, 1] / skf.n_splits
    test_mlp += mlp_model.predict_proba(X_fe_test)[:, 1] / skf.n_splits

ensemble_oof = (oof_lgb + oof_lr + oof_mlp) / 3
ensemble_test = (test_lgb + test_lr + test_mlp) / 3

print('LightGBM OOF AUC:', round(roc_auc_score(y_fe, oof_lgb), 4))
print('LogReg   OOF AUC:', round(roc_auc_score(y_fe, oof_lr), 4))
print('MLP      OOF AUC:', round(roc_auc_score(y_fe, oof_mlp), 4))
print('Ensemble OOF AUC:', round(roc_auc_score(y_fe, ensemble_oof), 4))

　単一モデルとアンサンブルの AUC を比較してみてください。

　アンサンブルは「複数のモデルの予測を組み合わせると安定しやすい」というメリットがありますが、**いつでもスコアが上がるとは限りません**。今回のように 1 つのモデル（LightGBM）が他を大きく上回る場合、単純平均すると弱いモデルに引きずられて、最強モデル単体より低いスコアになることがあります。

　単純平均がうまく働くのは、各モデルが**ある程度近い精度**を持ち、かつ**間違える場面が互いに違う**ときです。今回のような場面では、

- LightGBM 単体を最終予測に使う
- 重み付き平均（強いモデルの比率を上げる）にする
- スタッキングのような、より高度なアンサンブル手法を使う

といった選択肢があります。実際のコンペでも「とりあえずアンサンブルすれば良い」とは限らず、**OOF スコアを見ながら判断する** ことが大切です。

　今回は OOF AUC を見ると LightGBM 単体（0.82 前後）がアンサンブル（0.78 前後）より明確に高いので、**最終提出には LightGBM 単体の予測を使う** ことにします。これは実際のコンペでもよくある判断です。「アンサンブルしたから提出する」のではなく、**OOF を比較して最良のものを選ぶ** のが正しい流れです。

## 8. LightGBM のハイパーパラメータチューニング

　Section 6 の比較で LightGBM が最も良いスコアを出し、Section 7 の単純平均アンサンブルでは LightGBM 単体を超えられませんでした。そこで、**LightGBM のハイパーパラメータを少し動かして、単体性能をさらに伸ばせないか** を試します。

　**いつチューニングするか／何をチューニングするか** には流れがあります。

1. **特徴量を整えてからチューニングする**：特徴量を変えるたびに最適なハイパーパラメータも変わります。先に特徴量を固めてから最後にチューニングする方が手戻りが少ないです。
2. **最良のモデル 1 本だけをチューニングする**：今回は LightGBM が他を大きく上回ったので、これに集中します。3 モデルすべてを丁寧にチューニングするのは計算コストが見合わないことが多い。
3. **効きそうなハイパーパラメータから動かす**：LightGBM では `num_leaves`、`max_depth`、`min_data_in_leaf`、`learning_rate`、`n_estimators` あたりが代表的です。今回は過学習制御に直結する `max_depth` と `min_data_in_leaf` を振ります。`learning_rate` と `n_estimators` は計算量とのトレードオフが大きいため、ここでは固定しておきます。

　Section 3 で RandomForest に対して行ったのと同様に、`max_depth` と `min_data_in_leaf` の 2 つを 3 値ずつ振って、5-fold CV でスコアを比較します。

- **`max_depth`**：木の深さの上限。大きいほど複雑な分割ができるが、過学習しやすい
- **`min_data_in_leaf`**：1 つの葉に入る最小サンプル数。大きいほど分岐が控えめになり、過学習を抑える

In [ ]:
# max_depth × min_data_in_leaf を 3×3 = 9 通り試す
lgb_grid = {
    'max_depth': [3, 5, 7],
    'min_data_in_leaf': [10, 20, 50],
}

best_auc = -1.0
best_params = None
for max_depth in lgb_grid['max_depth']:
    for min_data_in_leaf in lgb_grid['min_data_in_leaf']:
        lgb_grid_model = lgb.LGBMClassifier(
            n_estimators=200,
            learning_rate=0.05,
            num_leaves=31,
            max_depth=max_depth,
            min_data_in_leaf=min_data_in_leaf,
            random_state=42,
            n_jobs=-1,
            verbose=-1,
        )
        scores = cross_val_score(lgb_grid_model, X_fe, y_fe, cv=skf, scoring='roc_auc')
        mean_auc = scores.mean()
        print(f'max_depth={max_depth}, min_data_in_leaf={min_data_in_leaf}: '
              f'mean CV AUC = {mean_auc:.4f}')
        if mean_auc > best_auc:
            best_auc = mean_auc
            best_params = {'max_depth': max_depth, 'min_data_in_leaf': min_data_in_leaf}

print()
print(f'Best params: {best_params}, Best CV AUC: {best_auc:.4f}')

　最も CV AUC が高かった設定が表示されました。今までの単一 LightGBM（CV AUC およそ 0.82）と比較して、改善が見られたかを確認します。

　次に、見つかったベストな設定で **全 train データを使って LightGBM を再学習** し、test 予測を作ります。

　**なぜ全データで再学習するのか？**　CV では各 fold の学習に train データの 4/5 しか使われていません。残りの 1/5 はその fold で検証用に取り分けられているためです。つまり CV の AUC は「4/5 のデータで学習したモデルの平均性能」を表しています。**最終提出時には全データを使って学習し直す** ことで、より多くの情報を反映したモデルになり、性能を最大限に引き出せます。CV はあくまで「どのハイパーパラメータが良いか」を判断する道具と位置付け、本番モデルは全データで作るのが定石です。

In [ ]:
# ベストな設定で、全 train データを使って学習
final_lgb = lgb.LGBMClassifier(
    n_estimators=200,
    learning_rate=0.05,
    num_leaves=31,
    max_depth=best_params['max_depth'],
    min_data_in_leaf=best_params['min_data_in_leaf'],
    random_state=42,
    n_jobs=-1,
    verbose=-1,
)
final_lgb.fit(X_fe, y_fe)
test_lgb_tuned = final_lgb.predict_proba(X_fe_test)[:, 1]

print(f'Final model trained on full data with params: {best_params}')

## 9. 予測の出力・提出

　Section 8 でチューニングした LightGBM を最終提出に使います。

In [ ]:
# Section 8 でチューニング・全train再学習した LightGBM の予測を採用
submission = pd.read_csv(PATH + 'sample_submission.csv')
submission["Drafted"] = test_lgb_tuned
submission.to_csv(PATH + 'submission_advanced.csv', index=False)
submission.head()

## 10. 今後のステップ

　本ノートブックでは、**EDA から始まり、前処理、ベースライン、クロスバリデーション、特徴量エンジニアリング、モデル比較、アンサンブルへと進む改善サイクル** を一通り体験しました。

　さらに精度を上げたい場合は、次のような方向が考えられます。

- `School` 列に Target Encoding やカウントエンコーディングを使う
- カテゴリ変数を One-Hot Encoding に変える
- 欠損値補完の方法を変える。たとえば中央値で埋める、グループ別の平均を使う、欠損自体を新しい特徴量にするなど
- LightGBM のハイパーパラメータを **Optuna** などのライブラリで自動探索する
- XGBoost や CatBoost など、他の勾配ブースティングのモデルを試す
- **スタッキング** に挑戦する。これは複数モデルの出力をさらに別のモデル（メタモデル）に学習させて、最終予測を作る手法
- ドメイン知識を反映した、より高度な派生特徴量を作る

　コンペでは、こうした打ち手を 1 つずつ検証して積み重ねていくことで、スコアの向上を狙っていきます。